In [ ]:
import pandas as pd
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error
# from sklearn-metric import mean_squared_error

In [57]:
url_jan_23 = 'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-01.parquet'
url_feb_23 = 'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-02.parquet'

In [58]:
df_train = pd.read_parquet(url_jan_23)
df_val = pd.read_parquet(url_feb_23)

# Q1

In [59]:
df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3066766 entries, 0 to 3066765
Data columns (total 19 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   VendorID               int64         
 1   tpep_pickup_datetime   datetime64[us]
 2   tpep_dropoff_datetime  datetime64[us]
 3   passenger_count        float64       
 4   trip_distance          float64       
 5   RatecodeID             float64       
 6   store_and_fwd_flag     object        
 7   PULocationID           int64         
 8   DOLocationID           int64         
 9   payment_type           int64         
 10  fare_amount            float64       
 11  extra                  float64       
 12  mta_tax                float64       
 13  tip_amount             float64       
 14  tolls_amount           float64       
 15  improvement_surcharge  float64       
 16  total_amount           float64       
 17  congestion_surcharge   float64       
 18  airport_fee           

A: 19

# Q2

In [60]:
df_train['duration'] = df_train.tpep_dropoff_datetime - df_train.tpep_pickup_datetime
df_train.duration = df_train.duration.apply(lambda td: td.total_seconds() / 60)

In [61]:
df_train.duration.std()

42.594351241920904

A: 42.59

# Q3

In [62]:
df_train_b = df_train[df_train.duration.between(1, 60)]

In [63]:
df_train.size

61335320

In [64]:
df_train_b.size

60183460

In [65]:
df_train_b.shape[0] * 100 / df_train.shape[0]

98.1220282212598

A: 98%

# Q4

In [66]:
df_train = df_train_b.copy()
df_train['PULocationID'] = df_train['PULocationID'].astype(str)
df_train['DOLocationID'] = df_train['DOLocationID'].astype(str)

In [67]:
categorical = ['PULocationID', 'DOLocationID'] #'PULocationID', 'DOLocationID']

dv = DictVectorizer()
target='duration'
train_dicts = df_train[categorical].to_dict(orient='records')
X_train = dv.fit_transform(train_dicts)
y_train = df_train[target].values
X_train.shape

(3009173, 515)

A: 515

# Q5

In [ ]:
lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred = lr.predict(X_train)

root_mean_squared_error(y_train, y_pred)
# mean_squared_error(y_train, y_pred, squared=False) --- IGNORE ---

7.649261932106969

A: 7.64

# Q6

In [69]:

df_val['duration'] = df_val.tpep_dropoff_datetime - df_val.tpep_pickup_datetime

# convert to minutes and filter outliers

df_val.duration = df_val.duration.apply(lambda td: td.total_seconds() / 60)

df_val = df_val[(df_val.duration >= 1) & (df_val.duration <= 60)]
df_val['PULocationID'] = df_val['PULocationID'].astype(str)
df_val['DOLocationID'] = df_val['DOLocationID'].astype(str)
categorical = ['PULocationID', 'DOLocationID'] #'PULocationID', 'DOLocationID']


val_dicts = df_val[categorical].to_dict(orient='records')
X_val = dv.transform(val_dicts)
target = 'duration'
y_val = df_val[target].values

In [ ]:
y_pred = lr.predict(X_val)

root_mean_squared_error(y_val, y_pred)
# mean_squared_error(y_val, y_pred, squared=False) --- IGNORE ---

7.811818743246608

A: 7.81